In [ ]:
### Pseudobulk coverage tracks
library(Seurat)
library(dplyr)

seurat_obj <- readRDS("/data/ebaird/scRNAseqreports/res/Gal10d_Gal12d_Flp10d_Flp12d_070525/seurat_with_regulons.rds")

In [ ]:
head(seurat_obj@meta.data$genotype)

In [ ]:
# Create genotype mapping
barcode_mapping <- data.frame(
  cell_barcode = colnames(seurat_obj),
  genotype = seurat_obj@meta.data$genotype,
  sample = seurat_obj@meta.data$condition
) %>%
  mutate(bam_barcode = gsub("_.*", "", cell_barcode))

# Create output directory if missing
if (!dir.exists("/data/ebaird/scRNAseqreports/res/Gal10d_Gal12d_Flp10d_Flp12d_070525/barcodes")) {
  dir.create("/data/ebaird/scRNAseqreports/res/Gal10d_Gal12d_Flp10d_Flp12d_070525/barcodes")
}

# Get valid sample-genotype pairs from data
valid_pairs <- barcode_mapping %>%
  distinct(genotype, sample) %>%
  filter(!is.na(genotype), !is.na(sample))

# Write only valid genotype-sample combinations
for (i in 1:nrow(valid_pairs)) {
  geno <- valid_pairs$genotype[i]
  samp <- valid_pairs$sample[i]
  
  barcodes <- barcode_mapping %>%
    filter(genotype == geno, sample == samp) %>%
    pull(bam_barcode)
  
  if (length(barcodes) > 0) {
    writeLines(barcodes, sprintf("/data/ebaird/scRNAseqreports/res/Gal10d_Gal12d_Flp10d_Flp12d_070525/barcodes/%s_%s.txt", geno, samp))
    message(sprintf("Writing %d barcodes for %s_%s", length(barcodes), geno, samp))
  } else {
    warning(sprintf("No cells found for VALID pair %s_%s", geno, samp))
  }
}

In [ ]:
head(barcode_mapping)

In [ ]:
library(Gviz)
library(rtracklayer)

# Define region
gene_region <- GRanges("")

# Create genotype tracks
gal_track <- DataTrack(
  range = "gal.bw",
  name = "gal",
  type = "h",
  col = "darkgreen",
  lwd = 2
)

flp_track <- DataTrack(
  range = "flp.bw",
  name = "flp",
  type = "h",
  col = "red",
  lwd = 2
)

# Add gene annotation
gene_track <- GeneRegionTrack(
  drosophila_gene_annotation,
  chromosome = "chr1",
  start = start(gene_region),
  end = end(gene_region),
  name = "Genes",
  transcriptAnnotation = "symbol"
)

# Plot
plotTracks(
  list(gal_track, flp_track, gene_track),
  chromosome = "chr1",
  from = start(gene_region),
  to = end(gene_region),
  title.width = 1.5,
  background.title = "white",
  groupAnnotation = "group"
)